# 🚀 Baidu Unlimited-OCR FastAPI Microservice on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaan7861/unlimited-ocr/blob/master/colab_unlimited_ocr.ipynb)

This notebook runs the **Baidu Unlimited-OCR** FastAPI microservice directly on Google Colab GPU.
- **Model Source**: Directly downloaded from Hugging Face (`baidu/Unlimited-OCR`).
- **Public Access**: Exposed via **Cloudflare Tunnel (`cloudflared`)** — **No account, sign-up, or auth tokens required!**

### Instructions:
1. **Set Runtime to GPU**: Click `Runtime` -> `Change runtime type` -> Select `T4 GPU` (or `A100` / `L4`) -> Click `Save`.
2. **Run All**: Click `Runtime` -> `Run all` (or press `Ctrl+F9`).

### Step 1: Verify NVIDIA GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Clone Repository

In [ ]:
import os

if not os.path.exists("app"):
    !git clone https://github.com/arslaan7861/unlimited-ocr.git
    %cd unlimited-ocr
else:
    print("Already inside repository root.")

### Step 3: Install Lightweight Dependencies & Cloudflared

In [ ]:
# Colab pre-installs PyTorch and CUDA. We only install web & OCR dependencies.
!pip install -q fastapi "uvicorn[standard]" pydantic-settings python-multipart pymupdf transformers addict einops easydict

# Download official Cloudflare Tunnel binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("Dependencies and cloudflared ready.")

### Step 4: Configure Production Environment for Hugging Face & CUDA

In [ ]:
env_content = """# Google Colab GPU Configuration
HOST=0.0.0.0
PORT=8000
ENVIRONMENT=production
LOG_LEVEL=INFO
MODEL_PATH=baidu/Unlimited-OCR
DEVICE=cuda
TORCH_DTYPE=bfloat16
MAX_UPLOAD_SIZE_MB=25
MAX_PDF_PAGES=50
MAX_CONCURRENT_INFERENCES=4
PDF_DPI=300
"""

with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)

print(".env configured to load model directly from Hugging Face: baidu/Unlimited-OCR")

### Step 5: Start FastAPI Server & Launch Cloudflare Tunnel

This will output your live, public HTTPS URL. You can use it in your frontend, CRM, or Postman!

In [ ]:
import re
import subprocess
import time

# 1. Start Cloudflare Tunnel in background
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# 2. Extract the public HTTPS trycloudflare.com URL
public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("=" * 65)
if public_url:
    print(f"🚀 PUBLIC API URL:     {public_url}")
    print(f"📖 INTERACTIVE DOCS:   {public_url}/docs")
    print(f"🩺 HEALTH PROBE:       {public_url}/health")
else:
    print("Tunnel started. Check output or run locally at http://localhost:8000")
print("=" * 65)

# 3. Start the FastAPI microservice with Uvicorn (downloads baidu/Unlimited-OCR on first load)
!uvicorn app.main:app --host 0.0.0.0 --port 8000